# Deep dive 3/13: `branches`

Only 350 rows -- small enough to eyeball directly (section 0 dumps the whole table), but still checked column-by-column since a small table is exactly where a subtle bug hides easiest (few rows means low odds of it standing out in a spot check).

Source of truth for what Silver *should* do: `data_pipelines/silver/table_specs.py`'s `branches_spec`. Source of truth for what the data *actually* looks like: this notebook, run against your real `data/latam_bank.duckdb`. Data dictionary declares 350 rows, Source: Internal, Partition: full_snapshot.

Run top to bottom. Checklist near the end; free cells after that.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. The whole table

350 rows is small enough to just look at directly. Worth a manual scan before the structured checks below -- sometimes a wrong value jumps out faster to a human eye than to a query you didn't think to write.

In [ ]:
q_full("SELECT * FROM silver.dim_branches ORDER BY branch_id")

## 1. Column inventory

In [ ]:
expected_silver_columns = [
    "branch_id", "branch_code", "branch_name", "branch_type", "address", "city", "state",
    "country", "postal_code", "geographic_zone", "phone", "email", "opening_time", "closing_time",
    "has_atms", "atm_count", "has_teller_windows", "teller_window_count", "latitude", "longitude",
    "branch_opening_date", "branch_status",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_branches'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.dim_branches:", missing or "none")
print("Extra in silver.dim_branches:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

In [ ]:
print("Bronze columns:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='branches' ORDER BY ordinal_position
"""))

print("Silver columns:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_branches' ORDER BY ordinal_position
"""))

## 2. Row-count reconciliation

In [ ]:
bronze_total = q("SELECT count(*) AS n FROM bronze.branches")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT branch_id) AS n FROM bronze.branches")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.dim_branches")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT branch_id) AS n FROM silver.dim_branches")["n"][0]

print(f"bronze.branches:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct branch_id")
print(f"silver.dim_branches:  {silver_total:,} rows, {silver_distinct_pk:,} distinct branch_id")
print()
print("Dictionary declares 350 rows -- matches" if bronze_total == 350 else f"Dictionary declares 350 rows -- MISMATCH, Bronze has {bronze_total}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

## 3. NOT NULL audit

Dictionary NOT NULL: `branch_id`, `branch_code`, `branch_name`, `branch_type`, `address`, `city`, `state`, `country`, `geographic_zone`, `phone`, `opening_time`, `closing_time`, `has_atms`, `has_teller_windows`, `branch_opening_date`, `branch_status`. Nullable: `postal_code`, `email`, `atm_count`, `teller_window_count`, `latitude`, `longitude`.

In [ ]:
declared_not_null = [
    "branch_id", "branch_code", "branch_name", "branch_type", "address", "city", "state",
    "country", "geographic_zone", "phone", "opening_time", "closing_time", "has_atms",
    "has_teller_windows", "branch_opening_date", "branch_status",
]

rows = []
total = q("SELECT count(*) AS n FROM bronze.branches")["n"][0]
for col in declared_not_null:
    nulls = q(f"""
        SELECT count(*) AS n FROM bronze.branches
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "actual_nulls": nulls,
                 "null_rate": round(nulls / total, 4) if total else None})

pd.DataFrame(rows)

## 4. UNIQUE constraint audit -- `branch_code`

Declared `NOT NULL, UNIQUE` in the dictionary. Not currently declared in `contracts.py` at all (unlike `document_number`/`product_number`, which now are after the earlier fix) -- worth knowing whether it's actually clean before deciding whether to add it.

In [ ]:
q("""
    SELECT
        count(*) AS total,
        count(DISTINCT branch_code) AS distinct_branch_code,
        count(*) FILTER (WHERE branch_code IS NULL OR trim(branch_code) = '') AS nulls
    FROM bronze.branches
""")

In [ ]:
q("""
    SELECT branch_code, count(*) AS n, count(DISTINCT branch_id) AS distinct_branch_ids
    FROM bronze.branches
    GROUP BY 1 HAVING count(*) > 1
    ORDER BY n DESC
""")

## 5. Domain checks

In [ ]:
# branch_type -- dictionary: Main, Express, Premium, Corporate
q("SELECT branch_type, count(*) AS n FROM bronze.branches GROUP BY 1 ORDER BY n DESC")

In [ ]:
# geographic_zone -- dictionary: Urban, Suburban, Rural
q("SELECT geographic_zone, count(*) AS n FROM bronze.branches GROUP BY 1 ORDER BY n DESC")

In [ ]:
# branch_status -- dictionary: Active, Temporarily Closed, Closed
q("SELECT branch_status, count(*) AS n FROM bronze.branches GROUP BY 1 ORDER BY n DESC")

## 6. Country canonicalization + state/country cross-check

Same pattern as `customers` -- `branches.country` also runs through `as_country()`.

In [ ]:
print("Distinct raw country values in Bronze:")
display(q("SELECT country, count(*) AS n FROM bronze.branches GROUP BY 1 ORDER BY n DESC"))

print("Distinct country values in Silver:")
display(q("SELECT country, count(*) AS n FROM silver.dim_branches GROUP BY 1 ORDER BY n DESC"))

country_canonical = {
    "méxico": "México", "mexico": "México", "colombia": "Colombia", "argentina": "Argentina",
    "usa": "USA", "united states": "USA", "spain": "Spain", "brazil": "Brazil", "brasil": "Brazil",
}
raw_values = q("SELECT DISTINCT lower(trim(country)) AS v FROM bronze.branches WHERE country IS NOT NULL")["v"].tolist()
uncovered = [v for v in raw_values if v not in country_canonical]
print("Raw country values NOT covered by COUNTRY_CANONICAL:", uncovered or "none -- full coverage")

In [ ]:
q("""
    SELECT country, state, count(*) AS n
    FROM silver.dim_branches
    GROUP BY 1, 2
    ORDER BY country, n DESC
""")

## 7. `postal_code` -- same VARCHAR-vs-numeric question as `customers`

In [ ]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE TRY_CAST(postal_code AS BIGINT) IS NOT NULL) AS numeric_parseable,
        round(100.0 * count(*) FILTER (WHERE TRY_CAST(postal_code AS BIGINT) IS NOT NULL) / count(*), 2) AS pct_numeric,
        count(*) FILTER (WHERE postal_code IS NULL OR trim(postal_code) = '') AS nulls
    FROM bronze.branches
""")

## 8. `phone`/`email` shape sanity -- same treatment as `customers`' contact fields

In [ ]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE phone IS NULL OR trim(phone) = '') AS phone_nulls,
        count(*) FILTER (WHERE phone IS NOT NULL AND trim(phone) <> ''
                          AND regexp_matches(phone, '[0-9]') = false) AS phone_no_digits,
        count(*) FILTER (WHERE email IS NULL OR trim(email) = '') AS email_nulls,
        count(*) FILTER (WHERE email IS NOT NULL AND trim(email) <> '' AND email NOT LIKE '%@%') AS email_missing_at
    FROM bronze.branches
""")

## 9. `opening_time`/`closing_time` -- parsed correctly, and internally consistent

`branches_spec` casts both with `transform="time"`. Beyond a successful cast, closing should be after opening for every branch (no branch that "closes before it opens").

In [ ]:
q("""
    SELECT
        typeof(opening_time), typeof(closing_time),
        count(*) FILTER (WHERE opening_time IS NULL) AS opening_nulls,
        count(*) FILTER (WHERE closing_time IS NULL) AS closing_nulls,
        count(*) FILTER (WHERE closing_time <= opening_time) AS closes_before_or_when_opens,
        min(opening_time) AS earliest_opening_time, max(opening_time) AS latest_opening_time,
        min(closing_time) AS earliest_closing_time, max(closing_time) AS latest_closing_time
    FROM silver.dim_branches
    GROUP BY ALL
""")

## 10. ATM / teller-window flag-vs-count consistency

`has_atms`/`has_teller_windows` are booleans; `atm_count`/`teller_window_count` are the corresponding integers. Nothing in Silver enforces they agree -- a branch could have `has_atms=False` with a positive `atm_count`, or `has_atms=True` with `atm_count` NULL/0. Checking both directions.

In [ ]:
print("Bronze raw has_atms values:")
display(q("SELECT typeof(has_atms), has_atms, count(*) AS n FROM bronze.branches GROUP BY ALL ORDER BY n DESC"))
print("Silver parsed has_atms values:")
display(q("SELECT typeof(has_atms), has_atms, count(*) AS n FROM silver.dim_branches GROUP BY ALL ORDER BY n DESC"))

In [ ]:
q("""
    SELECT
        has_atms,
        count(*) AS n,
        count(*) FILTER (WHERE atm_count IS NULL) AS atm_count_null,
        count(*) FILTER (WHERE atm_count = 0) AS atm_count_zero,
        count(*) FILTER (WHERE atm_count > 0) AS atm_count_positive,
        min(atm_count) AS min_atm_count, max(atm_count) AS max_atm_count
    FROM silver.dim_branches
    GROUP BY 1
    ORDER BY 1
""")

In [ ]:
print("Bronze raw has_teller_windows values:")
display(q("SELECT typeof(has_teller_windows), has_teller_windows, count(*) AS n FROM bronze.branches GROUP BY ALL ORDER BY n DESC"))
print("Silver parsed has_teller_windows values:")
display(q("SELECT typeof(has_teller_windows), has_teller_windows, count(*) AS n FROM silver.dim_branches GROUP BY ALL ORDER BY n DESC"))

In [ ]:
q("""
    SELECT
        has_teller_windows,
        count(*) AS n,
        count(*) FILTER (WHERE teller_window_count IS NULL) AS twc_null,
        count(*) FILTER (WHERE teller_window_count = 0) AS twc_zero,
        count(*) FILTER (WHERE teller_window_count > 0) AS twc_positive,
        min(teller_window_count) AS min_twc, max(teller_window_count) AS max_twc
    FROM silver.dim_branches
    GROUP BY 1
    ORDER BY 1
""")

## 11. `latitude`/`longitude` -- populated rate and plausible bounding box

México, Colombia, and Argentina together roughly span latitude -55 to 33 and longitude -118 to -34. Anything outside that box (or a suspicious (0, 0) "null island") is worth a second look rather than assuming the cast alone caught it.

In [ ]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE latitude IS NOT NULL AND longitude IS NOT NULL) AS both_populated,
        min(latitude) AS min_lat, max(latitude) AS max_lat,
        min(longitude) AS min_lon, max(longitude) AS max_lon,
        count(*) FILTER (WHERE latitude NOT BETWEEN -56 AND 34) AS lat_out_of_box,
        count(*) FILTER (WHERE longitude NOT BETWEEN -119 AND -33) AS lon_out_of_box,
        count(*) FILTER (WHERE latitude = 0 AND longitude = 0) AS null_island
    FROM silver.dim_branches
""")

In [ ]:
# Do the coordinates actually land inside their own declared country? A cheap proxy: each
# country's branches should cluster tightly, not spread across the whole bounding box.
q("""
    SELECT country,
        count(*) AS n,
        round(min(latitude), 2) AS min_lat, round(max(latitude), 2) AS max_lat,
        round(min(longitude), 2) AS min_lon, round(max(longitude), 2) AS max_lon
    FROM silver.dim_branches
    WHERE latitude IS NOT NULL AND longitude IS NOT NULL
    GROUP BY 1
""")

## 11.5 Longitude anomaly follow-up -- 167/350 branches (47.7%) have an implausible longitude

Section 11 found `lon_out_of_box = 167` and every country's `max_lon` landing on the exact same value, `0.099995` -- a coincidence too precise to be random noise. If nearly half the branches share one specific wrong value (rather than being scattered errors), that points to a single sentinel/default longitude standing in for a real one, not independent corruption. These cells isolate exactly which rows and whether they share anything else in common.

In [ ]:
# How many rows have exactly this value (or something extremely close to it, in case of float
# formatting noise), vs. how many are "out of box" but at some other bad value?
q("""
    SELECT
        count(*) FILTER (WHERE abs(longitude - 0.099995) < 0.0001) AS exact_sentinel_matches,
        count(*) FILTER (WHERE longitude NOT BETWEEN -119 AND -33 AND abs(longitude - 0.099995) >= 0.0001) AS other_out_of_box,
        count(*) FILTER (WHERE longitude NOT BETWEEN -119 AND -33) AS total_out_of_box
    FROM silver.dim_branches
""")

In [ ]:
# Full detail on the affected rows -- looking for any shared trait (branch_type, opening date,
# _source_file, whether latitude also looks wrong) that would point at a root cause.
q_full("""
    SELECT branch_id, branch_code, branch_name, country, city, latitude, longitude, branch_type, branch_opening_date
    FROM silver.dim_branches
    WHERE longitude NOT BETWEEN -119 AND -33
    ORDER BY country, city
""")

In [ ]:
# Is this isolated to longitude, or does latitude show the same pattern for these same rows?
# (i.e. did both coordinates get corrupted together, or just longitude?)
q("""
    SELECT
        count(*) AS affected_rows,
        min(latitude) AS min_lat_of_affected, max(latitude) AS max_lat_of_affected,
        count(*) FILTER (WHERE latitude NOT BETWEEN -56 AND 34) AS latitude_also_out_of_box
    FROM silver.dim_branches
    WHERE longitude NOT BETWEEN -119 AND -33
""")

In [ ]:
# Bronze-level check -- is the bad value already wrong in Bronze (a source-data defect) or does
# it only appear wrong in Silver (a transform bug)? Compares raw text to the cast double.
q_full("""
    SELECT b.branch_id, b.longitude AS bronze_raw_longitude, s.longitude AS silver_cast_longitude
    FROM bronze.branches b
    JOIN silver.dim_branches s USING (branch_id)
    WHERE s.longitude NOT BETWEEN -119 AND -33
    ORDER BY b.branch_id
    LIMIT 20
""")

## 12. `branch_opening_date` sanity

In [ ]:
q("""
    SELECT
        min(branch_opening_date) AS earliest_opening, max(branch_opening_date) AS latest_opening,
        count(*) FILTER (WHERE branch_opening_date > current_date) AS future_opening_dates,
        count(*) FILTER (WHERE branch_opening_date < DATE '2023-06-17') AS before_dataset_window,
        count(*) FILTER (WHERE branch_opening_date > DATE '2026-06-17') AS after_dataset_window
    FROM silver.dim_branches
""")

## 13. Known context

`branches.branch_id` is the parent side of the `registration_branch_id`/`assigned_branch_id` FK anomaly already fully diagnosed in `diagnose_branch_fk.ipynb` -- that investigation concluded the defect is in the *child* tables (`customers`, `service_agents`), not here. `products.opening_branch_id`, `transactions.branch_id`, and `complaints.related_branch_id`/`assigned_agent_id` were all confirmed to resolve cleanly against this table. Nothing in this notebook should change that conclusion; it's here for cross-reference only.

## Checklist

- [ ] Column inventory (1): spec/table agree
- [ ] Row reconciliation (2): 350 rows as the dictionary declares, no dedup mismatch
- [ ] NOT NULL audit (3): all 16 declared columns populated
- [ ] UNIQUE audit (4): `branch_code` genuinely unique, or note any real collision
- [ ] Domain checks (5): `branch_type`/`geographic_zone`/`branch_status` match declared sets
- [ ] Country canonicalization (6): full coverage, state/country pairs all plausible
- [ ] `postal_code` (7): note the numeric-parseable rate
- [ ] `phone`/`email` (8): note null rates and any structurally malformed values
- [ ] Time fields (9): no branch closes before/when it opens
- [ ] ATM/teller consistency (10): flag and count agree in both directions, or note where they don't
- [ ] Lat/long (11): no out-of-box coordinates, no null-island rows, each country's branches cluster sensibly
- [ ] `branch_opening_date` (12): no future dates, note any window mismatch (expected, per the `customers`/`products` precedent)

In [ ]:
con.close()
print("closed")